# 11 - Reranking Test candidates with AlignScore

## Initialization

In [8]:
import os
import math
import json
import pandas as pd
from pathlib import Path
import torch
import transformers
from tqdm.auto import tqdm
transformers.AdamW = torch.optim.AdamW
from alignscore import AlignScore

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TEST_PATH = ROOT / "data" / "processed" / "test.jsonl"
OUT_DIR = ROOT / "outputs/test"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUT_DIR / "overall_alignscore_results.csv"
CANDIDATE_SCORES_CSV = OUT_DIR / "candidate_alignscores.csv"
BASELINE_SCORES_CSV = OUT_DIR / "baseline_alignscores.csv"
DATA_CSV = OUT_DIR / "test_generated_data.csv"
BASELINE_CSV = ROOT / "outputs" / "test_baseline.csv"

assert TEST_PATH.exists(), f"Missing test dataset: {TEST_PATH}"
assert DATA_CSV.exists(), f"Missing generated candidates: {DATA_CSV}"
assert BASELINE_CSV.exists(), f"Missing baseline results: {BASELINE_CSV}"

def require(ok, message):
    if not ok:
        raise ValueError(message)

CONFIG = dict(model = 'roberta-base',
            evaluation_mode = 'nli_sp',
            batch_size = 8,
            checkpoint = 'checkpoints/AlignScore-base.ckpt')

ckpt = ROOT / CONFIG["checkpoint"]

require(ckpt.is_file(), f"Missing AlignScore checkpoint: {ckpt}")

os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

device = "cuda:0" if torch.cuda.is_available() else "cpu"


print(f"AlignScore device: {device}")
print(f"Loading AlignScore model from {ckpt}")

AlignScore device: cuda:0
Loading AlignScore model from c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\checkpoints\AlignScore-base.ckpt


## Loading test data and generated candidates

In [9]:
with open(TEST_PATH, "r", encoding="utf-8") as f:
    data = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

candidates_df = pd.read_csv(DATA_CSV)
baseline_df = pd.read_csv(BASELINE_CSV)

abstracts = {
    str(row["id"]): row["abstract"]
    for row in data
}

candidates = {
    str(row["id"]): row
    for _, row in candidates_df.iterrows()
}

baselines = {
    str(row["id"]): row
    for _, row in baseline_df.iterrows()
}

assert len(candidates) == len(data) == len(baselines), "Article count mismatch."

print("Test articles:", len(data))
print("Generated articles:", len(candidates))
print("Baseline articles:", len(baselines))
print("Expected candidate summaries:", len(data) * 5)

Test articles: 615
Generated articles: 615
Baseline articles: 615
Expected candidate summaries: 3075


## AlignScore Iniatialization

In [21]:
scorer = AlignScore(model = CONFIG["model"], 
                    batch_size = CONFIG["batch_size"], 
                    evaluation_mode = CONFIG["evaluation_mode"], 
                    ckpt_path = str(ckpt), 
                    device = device, 
                    verbose = False
                    )

print("AlignScore loaded successfully.")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\.venv-scoring\Lib\site-packages\lightning_fabric\utilities\cloud_io.py:51: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location)  # type: ignore[arg-type]
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\checkpoints\AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized

AlignScore loaded successfully.


## Scoring and reranking candidates

In [22]:
if OUT_CSV.exists():
    rows = pd.read_csv(OUT_CSV).to_dict("records")
    completed_ids = {str(r["id"]) for r in rows}
else:
    rows = []
    completed_ids = set()

for article in tqdm(data, desc="AlignScore Evaluation"):

    article_id = str(article["id"])

    if article_id in completed_ids:
        continue

    baseline_row = baselines[article_id]
    candidate_row = candidates[article_id]
    
    candidate_texts = [
        str(candidate_row[f"candidate_{j}"]).strip()
        for j in range(1, 6)
    ]
    baseline_texts = [
        str(baseline_row["generated_tldr"]).strip()
    ]
    assert all(candidate_texts), f"Empty candidate for {article_id}"
    assert all(baseline_texts), f"Empty baseline for {article_id}"

    with torch.inference_mode():
        candidate_scores = scorer.score(
            contexts=[article["abstract"]] * 5,
            claims=candidate_texts
        )

    candidate_scores = [float(score) for score in candidate_scores]

    with torch.inference_mode():
            baseline_scores= scorer.score(
                contexts=[article["abstract"]],
                claims=baseline_texts
            )
    
    baseline_scores = [float(score) for score in baseline_scores]

    assert len(candidate_scores) == 5
    assert len(baseline_scores) == 1
    assert all(math.isfinite(score) for score in candidate_scores)
    assert all(math.isfinite(score) for score in baseline_scores)

    best = max(range(5), key=lambda j: candidate_scores[j])

    rows.append({
        "id": article_id,
        "baseline_summary": baseline_row["generated_tldr"],
        "selected_summary": candidate_texts[best],
        "baseline_alignscore": baseline_scores[0],
        "selected_alignscore": candidate_scores[best],
        "selected_candidate": best + 1,
        **{f"alignscore_{j+1}": score
            for j, score in enumerate(candidate_scores)}
    })

    completed_ids.add(article_id)

    # Save progress after each article
    pd.DataFrame(rows).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

AlignScore Evaluation:   0%|          | 0/615 [00:00<?, ?it/s]

## Saving individual candidate scores

In [ ]:
inspect_inputs = []
inspect_baseline_inputs = []

reranked = {
    str(row["id"]): row
    for row in rows
}

for article_id, candidate_row in candidates.items():

    result = reranked.get(article_id)

    if result is None:
        continue

    for j in range(1, 6):
        inspect_inputs.append({
            "id": article_id,
            "candidate": j,
            "summary": candidate_row[f"candidate_{j}"],
            "alignscore": result[f"alignscore_{j}"]
        })

for article_id, baseline_row in baselines.items():

    result = reranked.get(article_id)

    if result is None:
        continue

    inspect_baseline_inputs.append({
        "id": article_id,
        "summary": baseline_row["generated_tldr"],
        "alignscore": result["baseline_alignscore"]
    })

candidate_scores_df = pd.DataFrame(inspect_inputs)
baseline_scores_df = pd.DataFrame(inspect_baseline_inputs)

candidate_scores_df.to_csv(
    CANDIDATE_SCORES_CSV,
    index=False,
    encoding="utf-8-sig"
)

baseline_scores_df.to_csv(
    BASELINE_SCORES_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Candidate scores:", len(candidate_scores_df))
print("Saved:", CANDIDATE_SCORES_CSV)
print("Baseline scores:", len(baseline_scores_df))
print("Saved:", BASELINE_SCORES_CSV)

Candidate scores: 3075
Saved: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\test\candidate_alignscores.csv
Baseline scores: 615
Saved: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\test\baseline_alignscores.csv


## Summary stats

In [24]:
candidate_summary = (
    candidate_scores_df.groupby("candidate")["alignscore"]
    .agg(["mean", "std", "min", "max"])
)
baseline_summary = (
    baseline_scores_df["alignscore"]
    .agg(["mean", "std", "min", "max"])
)

print(candidate_summary)
print("Baseline: \n" + str(baseline_summary))

print(
    "\nOverall candidate mean AlignScore:",
    candidate_scores_df["alignscore"].mean()
)

print(
    "Mean selected candidate AlignScore:",
    pd.DataFrame(rows)["selected_alignscore"].mean()
)

print(
    "Overall baseline mean AlignScore:",
    baseline_scores_df["alignscore"].mean()
)

               mean       std       min       max
candidate                                        
1          0.940243  0.171705  0.001858  0.998333
2          0.928733  0.187364  0.001705  0.998626
3          0.914079  0.187590  0.002783  0.998483
4          0.896945  0.215601  0.009410  0.998849
5          0.877579  0.227789  0.001236  0.998622
Baseline: 
mean    0.991045
std     0.036095
min     0.153619
max     0.998849
Name: alignscore, dtype: float64

Overall candidate mean AlignScore: 0.9115157696626687
Mean selected candidate AlignScore: 0.991044578610397
Overall baseline mean AlignScore: 0.991044578610397
